# 학급 내 특정 학생의 교사용 리포트 채점 함수

## 정체성 발달 및 친구관계

### 정체성 발달

In [ ]:
# 깊은 탐색 - 자기보고 103~105번
# 넓은 탐색 - 자기보고 94~96번
# 전념 결단 - 자기보고 91~93번
# 전념 동일시 - 자기보고 100~102번
# 반추적 탐색 - 자기보고 97~99번

In [38]:
# 지명데이터 먼저 불러와서 자기보고 데이터와 병합 전처리
import sys
print(sys.executable)   # 예: C:\ProgramData\anaconda3\python.exe
print(sys.path)         # 패키지 탐색 경로(여기에 site-packages가 있어야 import 가능)

import awswrangler as wr
query_df = wr.athena.read_sql_query(
    """
    select a.school_code, a.school_name, b.school_grade, b.school_num, b.class_name, c.psy_name, c.psy_code, c.close_yn, f.student_num, d.target_code, e.user_testing_no, e.question_no, e.student_code 
    from school_info a
    inner join school_class b on a.school_code = b.school_code
    inner join psy_class c on b.class_code = c.class_code
    inner join psy_target_detail d on c.class_code = d.class_code and c.psy_code = d.psy_code
    inner join at_user_testing_paper_pn e on d.user_testing_no = e.user_testing_no
    inner join school_student f on d.target_code = f.student_code and d.class_code = f.class_code
    where c.close_yn = 'Y' and e.question_no in (1, 4, 5, 6, 7, 12)
    and d.psy_code = 'P202511122059'
    """
    , database="schoolfriends-kmj"
)

c:\ProgramData\anaconda3\python.exe
['c:\\ProgramData\\anaconda3\\python313.zip', 'c:\\ProgramData\\anaconda3\\DLLs', 'c:\\ProgramData\\anaconda3\\Lib', 'c:\\ProgramData\\anaconda3', '', 'C:\\Users\\USER\\AppData\\Roaming\\Python\\Python313\\site-packages', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\win32', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\win32\\lib', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\Pythonwin']


In [39]:
query_df['user_testing_no'] = query_df['user_testing_no'].str.replace('"', '')
query_df

,school_code,school_name,school_grade,school_num,class_name,psy_name,psy_code,close_yn,student_num,target_code,user_testing_no,question_no,student_code
0,AD0001,학지중학교,<NA>,<NA>,"""""","""25년 11월_1차""",P202511122059,Y,<NA>,AD00012025115D1E4020,8916207158814a67abc2,1,AD00012025115D1E4010
1,AD0001,학지중학교,<NA>,<NA>,"""""","""25년 11월_1차""",P202511122059,Y,<NA>,AD00012025115D1E4020,8916207158814a67abc2,1,AD00012025115D1E4016
2,AD0001,학지중학교,<NA>,<NA>,"""""","""25년 11월_1차""",P202511122059,Y,<NA>,AD00012025115D1E4020,8916207158814a67abc2,4,AD00012025115D1E4003
3,AD0001,학지중학교,<NA>,<NA>,"""""","""25년 11월_1차""",P202511122059,Y,<NA>,AD00012025115D1E4020,8916207158814a67abc2,4,AD00012025115D1E4008
4,AD0001,학지중학교,<NA>,<NA>,"""""","""25년 11월_1차""",P202511122059,Y,<NA>,AD00012025115D1E4020,8916207158814a67abc2,4,AD00012025115D1E4010
...,...,...,...,...,...,...,...,...,...,...,...,...,...
243,AD0001,학지중학교,<NA>,<NA>,"""""","""25년 11월_1차""",P202511122059,Y,<NA>,AD00012025115D1E4001,5021a196f7c74c81a151,12,AD00012025115D1E4007
244,AD0001,학지중학교,<NA>,<NA>,"""""","""25년 11월_1차""",P202511122059,Y,<NA>,AD00012025115D1E4001,5021a196f7c74c81a151,12,AD00012025115D1E4009
245,AD0001,학지중학교,<NA>,<NA>,"""""","""25년 11월_1차""",P202511122059,Y,<NA>,AD00012025115D1E4001,5021a196f7c74c81a151,12,AD00012025115D1E4010
246,AD0001,학지중학교,<NA>,<NA>,"""""","""25년 11월_1차""",P202511122059,Y,<NA>,AD00012025115D1E4001,5021a196f7c74c81a151,12,AD00012025115D1E4012


In [40]:
# 지명수 구하는 함수
def count_point(df, student_list, item_no) :
    import pandas as pd
    if df['psy_code'].nunique() > 1 :
        raise ValueError('psy_code 컬럼에 여러 값이 존재합니다.')
    else :
        temp_df = df[df['question_no'].isin(item_no)] # 해당 문항 필터링
        count_list = []
        for student in student_list :
            if len(temp_df[temp_df['student_code']==student]) >= 1 :
                freq = len(temp_df[temp_df['student_code']==student])
            elif len(temp_df[temp_df['student_code']==student]) == 0 :
                freq = 0
            count_list.append(freq)
        count_point_df = pd.DataFrame({'student_code': student_list, 'num_point': count_list})
        return count_point_df, df['psy_code'].iloc[0]

# 학생 리스트 구하는 함수
def student_list(df) :
    student_list = df['target_code'].unique().tolist()
    return student_list, df['psy_code'].iloc[0]

In [41]:
# 검수하고자 하는 평가명 필터링
df = query_df[query_df['psy_name'].str.contains('25년 11월_1차')]
print(df)

    school_code school_name  school_grade  school_num class_name  \
0        AD0001       학지중학교          <NA>        <NA>         ""   
1        AD0001       학지중학교          <NA>        <NA>         ""   
2        AD0001       학지중학교          <NA>        <NA>         ""   
3        AD0001       학지중학교          <NA>        <NA>         ""   
4        AD0001       학지중학교          <NA>        <NA>         ""   
..          ...         ...           ...         ...        ...   
243      AD0001       학지중학교          <NA>        <NA>         ""   
244      AD0001       학지중학교          <NA>        <NA>         ""   
245      AD0001       학지중학교          <NA>        <NA>         ""   
246      AD0001       학지중학교          <NA>        <NA>         ""   
247      AD0001       학지중학교          <NA>        <NA>         ""   

         psy_name       psy_code close_yn  student_num           target_code  \
0    "25년 11월_1차"  P202511122059        Y         <NA>  AD00012025115D1E4020   
1    "25년 11월_1차"  P202

In [15]:
# 평가 내역 내의 고유의 학생 코드와 테스팅 넘버
unique_list_df = df.loc[:,['target_code', 'user_testing_no']].drop_duplicates(subset=['target_code', 'user_testing_no'])
print(unique_list_df)

              target_code       user_testing_no
0    AD00012025115D1E4020  8916207158814a67abc2
10   AD00012025115D1E4019  a28c9553672c4f54ad7d
20   AD00012025115D1E4018  157327d769394633811f
35   AD00012025115D1E4017  d163f8c83ef64c50aa87
46   AD00012025115D1E4016  ac181f9b8eaa426cb4c8
57   AD00012025115D1E4015  72559204bbdf448b912f
68   AD00012025115D1E4014  30aa23408f104d1293a4
85   AD00012025115D1E4013  4f3f46fc2d2244758a78
97   AD00012025115D1E4012  dc9998566c234f0599c3
111  AD00012025115D1E4011  1eb4ec746358472a9a97
123  AD00012025115D1E4010  a511507acce04b908351
139  AD00012025115D1E4009  14c8df44e4004885a5e0
150  AD00012025115D1E4008  1d49e9937ff141f98e40
159  AD00012025115D1E4007  083269d802e940ce95be
170  AD00012025115D1E4006  e0cebb8191764f41a106
189  AD00012025115D1E4005  bd5e577fd48840d6bd34
198  AD00012025115D1E4004  521595a387b449f8b55d
211  AD00012025115D1E4003  59237a4bfe5b46189c25
225  AD00012025115D1E4002  62dfbfb230e1497c963a
234  AD00012025115D1E4001  5021a196f7c74

In [16]:
# 타입오류가 계속 발생하여
# 급하기 때문에 raw csv에서 학생명단으로 병합하여 검수 진행
import pandas as pd
testing_paper_df = pd.read_csv("AT_USER_TESTING_PAPER_202511140921.csv")
# 고유 학생 리스트와 병합하여 자기보고 json 추출
testing_paper_df_filtered = pd.merge(testing_paper_df, unique_list_df, left_on='USER_TESTING_NO', right_on='user_testing_no', how='inner')
print(testing_paper_df_filtered)

         USER_TESTING_NO                                         PAPER_JSON  \
0   083269d802e940ce95be  {"questionChoiceList":[{"questionNo":1,"choice...   
1   14c8df44e4004885a5e0  {"questionChoiceList":[{"questionNo":1,"choice...   
2   157327d769394633811f  {"questionChoiceList":[{"questionNo":1,"choice...   
3   1d49e9937ff141f98e40  {"questionChoiceList":[{"questionNo":1,"choice...   
4   1eb4ec746358472a9a97  {"questionChoiceList":[{"questionNo":1,"choice...   
5   30aa23408f104d1293a4  {"questionChoiceList":[{"questionNo":1,"choice...   
6   4f3f46fc2d2244758a78  {"questionChoiceList":[{"questionNo":1,"choice...   
7   5021a196f7c74c81a151  {"questionChoiceList":[{"questionNo":1,"choice...   
8   521595a387b449f8b55d  {"questionChoiceList":[{"questionNo":1,"choice...   
9   59237a4bfe5b46189c25  {"questionChoiceList":[{"questionNo":1,"choice...   
10  62dfbfb230e1497c963a  {"questionChoiceList":[{"questionNo":1,"choice...   
11  72559204bbdf448b912f  {"questionChoiceList":[{"q

In [19]:
import json
self_report_df = pd.DataFrame()
for idx, row in testing_paper_df_filtered.iterrows():
    user_testing_no = row['USER_TESTING_NO']
    student_code = row['target_code']
    json_data = json.loads(row['PAPER_JSON'])
    question_choices_df = pd.DataFrame(json_data['questionChoiceList'])
    question_choices_df = question_choices_df['choiceScore']
    temp_df = pd.DataFrame(question_choices_df).transpose()
    temp_df['USER_TESTING_NO'] = user_testing_no
    temp_df['STUDENT_CODE'] = student_code
    self_report_df = pd.concat([self_report_df, temp_df], ignore_index=True)
print(self_report_df)

    0  1  2  3  4  5  6  7  8  9  ... 109 110 111 112 113 114 115 116  \
0   1  4  3  3  3  3  3  3  1  1  ...   4   4   4   3   1   4   3   3   
1   4  2  3  3  1  3  3  5  4  3  ...   4   3   4   3   5   5   4   5   
2   3  3  3  3  1  3  3  3  2  4  ...   1   5   5   4   4   3   5   5   
3   4  5  5  2  1  1  1  3  4  3  ...   5   2   2   1   1   2   3   1   
4   4  5  3  4  3  3  1  2  1  1  ...   5   1   3   2   1   1   2   1   
5   4  5  5  5  2  5  2  4  5  3  ...   1   3   1   3   1   2   2   1   
6   3  4  4  2  3  3  4  4  5  5  ...   5   5   5   3   2   5   3   4   
7   5  5  2  5  3  4  4  4  4  3  ...   4   1   3   3   3   3   2   1   
8   1  5  5  4  4  2  2  1  3  3  ...   5   5   5   5   2   5   3   4   
9   2  1  1  2  1  1  2  3  3  2  ...   1   1   3   5   1   1   2   3   
10  3  3  5  5  5  3  4  3  1  5  ...   2   5   2   1   5   1   3   1   
11  3  3  3  1  2  3  2  2  1  4  ...   5   4   3   3   2   3   5   4   
12  5  3  3  4  4  5  5  4  3  4  ...   2   3   3  

In [22]:
# 컬럼명 변경
self_report_df.columns = [f'q{col+1}' if isinstance(col, int) else col for col in self_report_df.columns]
# 컬럼명 순서 변경
self_report_df = self_report_df[['USER_TESTING_NO', 'STUDENT_CODE'] + [col for col in self_report_df.columns if col not in  ['USER_TESTING_NO', 'STUDENT_CODE']]]
self_report_df.sort_values(by=['STUDENT_CODE'], inplace=True)
print(self_report_df)

         USER_TESTING_NO          STUDENT_CODE q1 q2 q3 q4 q5 q6 q7 q8  ...  \
7   5021a196f7c74c81a151  AD00012025115D1E4001  5  5  2  5  3  4  4  4  ...   
10  62dfbfb230e1497c963a  AD00012025115D1E4002  3  3  5  5  5  3  4  3  ...   
9   59237a4bfe5b46189c25  AD00012025115D1E4003  2  1  1  2  1  1  2  3  ...   
8   521595a387b449f8b55d  AD00012025115D1E4004  1  5  5  4  4  2  2  1  ...   
16  bd5e577fd48840d6bd34  AD00012025115D1E4005  1  3  2  2  1  1  1  3  ...   
19  e0cebb8191764f41a106  AD00012025115D1E4006  2  4  2  1  1  1  5  5  ...   
0   083269d802e940ce95be  AD00012025115D1E4007  1  4  3  3  3  3  3  3  ...   
3   1d49e9937ff141f98e40  AD00012025115D1E4008  4  5  5  2  1  1  1  3  ...   
1   14c8df44e4004885a5e0  AD00012025115D1E4009  4  2  3  3  1  3  3  5  ...   
14  a511507acce04b908351  AD00012025115D1E4010  1  1  1  1  1  1  4  3  ...   
4   1eb4ec746358472a9a97  AD00012025115D1E4011  4  5  3  4  3  3  1  2  ...   
18  dc9998566c234f0599c3  AD00012025115D1E4012  3  5

In [28]:
# 깊은 탐색 - 자기보고 103~105번
# 넓은 탐색 - 자기보고 94~96번
# 전념 결단 - 자기보고 91~93번
# 전념 동일시 - 자기보고 100~102번
# 반추적 탐색 - 자기보고 97~99번
# 타입변환
self_report_df[[col for col in self_report_df.columns if col.startswith('q')]] = self_report_df[[col for col in self_report_df.columns if col.startswith('q')]].apply(pd.to_numeric, errors='coerce')
# 행별 평균 구하기
self_report_df['깊은 탐색'] = self_report_df[['q103', 'q104', 'q105']].mean(axis=1).round()
self_report_df['넓은 탐색'] = self_report_df[['q94', 'q95', 'q96']].mean(axis=1).round()
self_report_df['전념 결단'] = self_report_df[['q91', 'q92', 'q93']].mean(axis=1).round()
self_report_df['전념 동일시'] = self_report_df[['q100', 'q101', 'q102']].mean(axis=1).round()
self_report_df['반추적 탐색'] = self_report_df[['q97', 'q98', 'q99']].mean(axis=1).round()
print(self_report_df)

         USER_TESTING_NO          STUDENT_CODE  q1  q2  q3  q4  q5  q6  q7  \
7   5021a196f7c74c81a151  AD00012025115D1E4001   5   5   2   5   3   4   4   
10  62dfbfb230e1497c963a  AD00012025115D1E4002   3   3   5   5   5   3   4   
9   59237a4bfe5b46189c25  AD00012025115D1E4003   2   1   1   2   1   1   2   
8   521595a387b449f8b55d  AD00012025115D1E4004   1   5   5   4   4   2   2   
16  bd5e577fd48840d6bd34  AD00012025115D1E4005   1   3   2   2   1   1   1   
19  e0cebb8191764f41a106  AD00012025115D1E4006   2   4   2   1   1   1   5   
0   083269d802e940ce95be  AD00012025115D1E4007   1   4   3   3   3   3   3   
3   1d49e9937ff141f98e40  AD00012025115D1E4008   4   5   5   2   1   1   1   
1   14c8df44e4004885a5e0  AD00012025115D1E4009   4   2   3   3   1   3   3   
14  a511507acce04b908351  AD00012025115D1E4010   1   1   1   1   1   1   4   
4   1eb4ec746358472a9a97  AD00012025115D1E4011   4   5   3   4   3   3   1   
18  dc9998566c234f0599c3  AD00012025115D1E4012   3   5   5   3  

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37428\1188738562.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self_report_df['넓은 탐색'] = self_report_df[['q94', 'q95', 'q96']].mean(axis=1).round()
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37428\1188738562.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self_report_df['전념 결단'] = self_report_df[['q91', 'q92', 'q93']].mean(axis=1).round()
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37428\1188738562.py:12: PerformanceWarning: DataFrame is

In [29]:
self_report_df

,USER_TESTING_NO,STUDENT_CODE,q1,q2,q3,q4,q5,q6,q7,q8,...,q113,q114,q115,q116,q117,깊은 탐색,넓은 탐색,전념 결단,전념 동일시,반추적 탐색
7,5021a196f7c74c81a151,AD00012025115D1E4001,5,5,2,5,3,4,4,4,...,3,3,3,2,1,4.0,3.0,4.0,4.0,3.0
10,62dfbfb230e1497c963a,AD00012025115D1E4002,3,3,5,5,5,3,4,3,...,1,5,1,3,1,5.0,3.0,4.0,4.0,4.0
9,59237a4bfe5b46189c25,AD00012025115D1E4003,2,1,1,2,1,1,2,3,...,5,1,1,2,3,2.0,4.0,2.0,2.0,1.0
8,521595a387b449f8b55d,AD00012025115D1E4004,1,5,5,4,4,2,2,1,...,5,2,5,3,4,4.0,3.0,4.0,4.0,5.0
16,bd5e577fd48840d6bd34,AD00012025115D1E4005,1,3,2,2,1,1,1,3,...,1,2,3,3,3,4.0,1.0,4.0,2.0,4.0
19,e0cebb8191764f41a106,AD00012025115D1E4006,2,4,2,1,1,1,5,5,...,2,3,1,2,2,2.0,4.0,3.0,2.0,2.0
0,083269d802e940ce95be,AD00012025115D1E4007,1,4,3,3,3,3,3,3,...,3,1,4,3,3,3.0,4.0,2.0,3.0,3.0
3,1d49e9937ff141f98e40,AD00012025115D1E4008,4,5,5,2,1,1,1,3,...,1,1,2,3,1,1.0,4.0,2.0,2.0,2.0
1,14c8df44e4004885a5e0,AD00012025115D1E4009,4,2,3,3,1,3,3,5,...,3,5,5,4,5,3.0,4.0,3.0,4.0,5.0
14,a511507acce04b908351,AD00012025115D1E4010,1,1,1,1,1,1,4,3,...,1,1,1,3,1,2.0,3.0,2.0,2.0,2.0


### 친구들과 맺고 있는 관계

In [43]:
# 앞에서 불러온 지명 데이터
# 지명데이터 먼저 불러와서 자기보고 데이터와 병합 전처리
import sys
print(sys.executable)   # 예: C:\ProgramData\anaconda3\python.exe
print(sys.path)         # 패키지 탐색 경로(여기에 site-packages가 있어야 import 가능)

import awswrangler as wr
query_df = wr.athena.read_sql_query(
    """
    select a.school_code, a.school_name, b.school_grade, b.school_num, b.class_name, c.psy_name, c.psy_code, c.close_yn, f.student_num, d.target_code, e.user_testing_no, e.question_no, e.student_code 
    from school_info a
    inner join school_class b on a.school_code = b.school_code
    inner join psy_class c on b.class_code = c.class_code
    inner join psy_target_detail d on c.class_code = d.class_code and c.psy_code = d.psy_code
    inner join at_user_testing_paper_pn e on d.user_testing_no = e.user_testing_no
    inner join school_student f on d.target_code = f.student_code and d.class_code = f.class_code
    where c.close_yn = 'Y' and e.question_no in (2,3,7,12,14,15)
    and d.psy_code = 'P202511122059'
    """
    , database="schoolfriends-kmj"
)
# 필요한 지명 문항 번호만 수정해서 쿼링
query_df['user_testing_no'] = query_df['user_testing_no'].str.replace('"', '')
df = query_df[query_df['psy_name'].str.contains('25년 11월_1차')]

c:\ProgramData\anaconda3\python.exe
['c:\\ProgramData\\anaconda3\\python313.zip', 'c:\\ProgramData\\anaconda3\\DLLs', 'c:\\ProgramData\\anaconda3\\Lib', 'c:\\ProgramData\\anaconda3', '', 'C:\\Users\\USER\\AppData\\Roaming\\Python\\Python313\\site-packages', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\win32', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\win32\\lib', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\Pythonwin']


In [44]:
# 각 척도별 지명수 구하기
scale_item_dict = {
    '친한 친구': [12],
    '친하고 싶은': [15],
    '서로 돕는': [7],
    '함께 공부하는': [14],
    '좋아하는': [2],
    '싫어하는': [3]
}
result_df = pd.DataFrame({'student_code': student_list(df)[0]})
for scale, items in scale_item_dict.items() :
    scale_point_df, psy_code = count_point(df, student_list(df)[0], items)
    scale_point_df.rename(columns={'num_point': f'{scale}_지명수'}, inplace=True)
    result_df = pd.merge(result_df, scale_point_df, on='student_code', how='left')
print(result_df)

            student_code  친한 친구_지명수  친하고 싶은_지명수  서로 돕는_지명수  함께 공부하는_지명수  \
0   AD00012025115D1E4020          2           6          9            3   
1   AD00012025115D1E4019          0           0          1            1   
2   AD00012025115D1E4018          4           1          9            1   
3   AD00012025115D1E4017          0           0          0            1   
4   AD00012025115D1E4016          8           5          6            3   
5   AD00012025115D1E4015          0           1          2           12   
6   AD00012025115D1E4014          0           1          0            3   
7   AD00012025115D1E4013          4           3          0            1   
8   AD00012025115D1E4012          7           8          6            2   
9   AD00012025115D1E4011          0           0          0            1   
10  AD00012025115D1E4010          3           3          0            0   
11  AD00012025115D1E4009          6           3          0            4   
12  AD00012025115D1E4008 

## 개인 척도와 학급 평균 비교 영역

### 또래지위와 사회적 행동

In [77]:
import sys
print(sys.executable)   # 예: C:\ProgramData\anaconda3\python.exe
print(sys.path)         # 패키지 탐색 경로(여기에 site-packages가 있어야 import 가능)

import awswrangler as wr
query_df = wr.athena.read_sql_query(
    """
    select a.school_code, a.school_name, b.school_grade, b.school_num, b.class_name, c.psy_name, c.psy_code, c.close_yn, f.student_num, d.target_code, e.user_testing_no, e.question_no, e.student_code 
    from school_info a
    inner join school_class b on a.school_code = b.school_code
    inner join psy_class c on b.class_code = c.class_code
    inner join psy_target_detail d on c.class_code = d.class_code and c.psy_code = d.psy_code
    inner join at_user_testing_paper_pn e on d.user_testing_no = e.user_testing_no
    inner join school_student f on d.target_code = f.student_code and d.class_code = f.class_code
    where c.close_yn = 'Y' and e.question_no in (1,2,4,5,6,7,8,9,10,11)
    and d.psy_code = 'P202511122059'
    """
    , database="schoolfriends-kmj"
)
query_df['user_testing_no'] = query_df['user_testing_no'].str.replace('"', '')
# 검수하고자 하는 평가명 필터링
df = query_df[query_df['psy_name'].str.contains('25년 11월_1차')]
print(df)

c:\ProgramData\anaconda3\python.exe
['c:\\ProgramData\\anaconda3\\python313.zip', 'c:\\ProgramData\\anaconda3\\DLLs', 'c:\\ProgramData\\anaconda3\\Lib', 'c:\\ProgramData\\anaconda3', '', 'C:\\Users\\USER\\AppData\\Roaming\\Python\\Python313\\site-packages', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\win32', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\win32\\lib', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\Pythonwin']
    school_code school_name  school_grade  school_num class_name  \
0        AD0001       학지중학교          <NA>        <NA>         ""   
1        AD0001       학지중학교          <NA>        <NA>         ""   
2        AD0001       학지중학교          <NA>        <NA>         ""   
3        AD0001       학지중학교          <NA>        <NA>         ""   
4        AD0001       학지중학교          <NA>        <NA>         ""   
..          ...         ...           ...         ...        ...   
426      AD0001       학지중학교         

In [78]:
# 지명 문항 활용하는 척도 전처리
point_scale_dict = {
    '인기' : [1],
    '선호' : [2],
    '주도적 공격성' : [4],
    '반응적 공격성' : [5],
    '친사회성' : [6,7],
    '가해' : [8,9],
    '피해' : [10,11]
}
result_point_scale_df = df.loc[:, ['target_code', 'user_testing_no']].copy().drop_duplicates(subset=['target_code', 'user_testing_no'])
for scale, items in point_scale_dict.items() :
    scale_point_df, psy_code = count_point(df, student_list(df)[0], items)
    scale_point_df.rename(columns={'num_point': f'{scale}_지명수'}, inplace=True)
    result_point_scale_df = pd.merge(result_point_scale_df, scale_point_df, left_on='target_code', right_on='student_code', how='left')
    result_point_scale_df.drop(columns=['student_code'], inplace=True)
result_point_scale_df

,target_code,user_testing_no,인기_지명수,선호_지명수,주도적 공격성_지명수,반응적 공격성_지명수,친사회성_지명수,가해_지명수,피해_지명수
0,AD00012025115D1E4020,8916207158814a67abc2,5,9,2,0,15,1,0
1,AD00012025115D1E4019,a28c9553672c4f54ad7d,0,0,5,3,1,15,0
2,AD00012025115D1E4018,157327d769394633811f,2,7,0,0,18,1,16
3,AD00012025115D1E4017,d163f8c83ef64c50aa87,0,0,1,10,2,0,12
4,AD00012025115D1E4016,ac181f9b8eaa426cb4c8,7,3,0,1,13,0,0
5,AD00012025115D1E4015,72559204bbdf448b912f,0,0,0,1,2,1,17
6,AD00012025115D1E4014,30aa23408f104d1293a4,0,0,0,0,0,1,0
7,AD00012025115D1E4013,4f3f46fc2d2244758a78,7,6,0,0,0,15,1
8,AD00012025115D1E4012,dc9998566c234f0599c3,7,4,2,0,13,1,20
9,AD00012025115D1E4011,1eb4ec746358472a9a97,0,0,0,0,0,0,2


In [79]:
# 자기보고 문항 전처리하여 병합
self_scale_dict = {
    '사회적 발달 목표' : ['q1', 'q2', 'q3', 'q4', 'q5', 'q6'],
    '사회적 과시 목표' : ['q7', 'q8', 'q9', 'q10', 'q11', 'q12'],
    '인지적 공감' : ['q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19'],
    '정서적 공감' : ['q20', 'q21', 'q22', 'q23', 'q24', 'q25', 'q26'],
    '도덕적 해이' : ['q70', 'q71', 'q72', 'q73', 'q74', 'q75', 'q76', 'q77', 'q78', 'q79', 'q80', 'q81', 'q82', 'q83'],
    '또래 동조성' : ['q106', 'q107', 'q108', 'q109', 'q110', 'q111', 'q112', 'q113', 'q114', 'q115', 'q116', 'q117'],
    '내재화' : ['q27', 'q28', 'q29', 'q30', 'q31', 'q32', 'q33', 'q34', 'q35', 'q36'],
    '행동적 수업 참여' : ['q45', 'q46', 'q47', 'q48', 'q49'],
    '정서적 수업 참여' : ['q50', 'q51', 'q52', 'q53', 'q54'],
    '발표 불안' : ['q84', 'q85', 'q86', 'q87', 'q88', 'q89', 'q90'],
    '협력적 태도' : [f'q{n}' for n in range(55, 62)],
    '경쟁적 태도' : [f'q{n}' for n in range(62, 70)]
}
self_scale_dict

{'사회적 발달 목표': ['q1', 'q2', 'q3', 'q4', 'q5', 'q6'],
 '사회적 과시 목표': ['q7', 'q8', 'q9', 'q10', 'q11', 'q12'],
 '인지적 공감': ['q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19'],
 '정서적 공감': ['q20', 'q21', 'q22', 'q23', 'q24', 'q25', 'q26'],
 '도덕적 해이': ['q70',
  'q71',
  'q72',
  'q73',
  'q74',
  'q75',
  'q76',
  'q77',
  'q78',
  'q79',
  'q80',
  'q81',
  'q82',
  'q83'],
 '또래 동조성': ['q106',
  'q107',
  'q108',
  'q109',
  'q110',
  'q111',
  'q112',
  'q113',
  'q114',
  'q115',
  'q116',
  'q117'],
 '내재화': ['q27', 'q28', 'q29', 'q30', 'q31', 'q32', 'q33', 'q34', 'q35', 'q36'],
 '행동적 수업 참여': ['q45', 'q46', 'q47', 'q48', 'q49'],
 '정서적 수업 참여': ['q50', 'q51', 'q52', 'q53', 'q54'],
 '발표 불안': ['q84', 'q85', 'q86', 'q87', 'q88', 'q89', 'q90'],
 '협력적 태도': ['q55', 'q56', 'q57', 'q58', 'q59', 'q60', 'q61'],
 '경쟁적 태도': ['q62', 'q63', 'q64', 'q65', 'q66', 'q67', 'q68', 'q69']}

In [80]:
# 급하기 때문에 raw csv에서 학생명단으로 병합하여 검수 진행
import pandas as pd
testing_paper_df = pd.read_csv("AT_USER_TESTING_PAPER_202511140921.csv")
# 학생 리스트 구하기
student_testing_list = df['user_testing_no'].unique().tolist()
print(student_testing_list)
print(len(student_testing_list))
# 병합하여 자기보고 json 추출
testing_paper_df_filtered = pd.merge(testing_paper_df, pd.DataFrame({'USER_TESTING_NO': student_testing_list}), on='USER_TESTING_NO', how='inner')
print(testing_paper_df_filtered)

['8916207158814a67abc2', 'a28c9553672c4f54ad7d', '157327d769394633811f', 'd163f8c83ef64c50aa87', 'ac181f9b8eaa426cb4c8', '72559204bbdf448b912f', '30aa23408f104d1293a4', '4f3f46fc2d2244758a78', 'dc9998566c234f0599c3', '1eb4ec746358472a9a97', 'a511507acce04b908351', '14c8df44e4004885a5e0', '1d49e9937ff141f98e40', '083269d802e940ce95be', 'e0cebb8191764f41a106', 'bd5e577fd48840d6bd34', '521595a387b449f8b55d', '59237a4bfe5b46189c25', '62dfbfb230e1497c963a', '5021a196f7c74c81a151']
20
         USER_TESTING_NO                                         PAPER_JSON  \
0   083269d802e940ce95be  {"questionChoiceList":[{"questionNo":1,"choice...   
1   14c8df44e4004885a5e0  {"questionChoiceList":[{"questionNo":1,"choice...   
2   157327d769394633811f  {"questionChoiceList":[{"questionNo":1,"choice...   
3   1d49e9937ff141f98e40  {"questionChoiceList":[{"questionNo":1,"choice...   
4   1eb4ec746358472a9a97  {"questionChoiceList":[{"questionNo":1,"choice...   
5   30aa23408f104d1293a4  {"questionChoice

In [81]:
self_report_df = pd.DataFrame()
for idx, row in testing_paper_df_filtered.iterrows():
    user_testing_no = row['USER_TESTING_NO']
    json_data = json.loads(row['PAPER_JSON'])
    question_choices_df = pd.DataFrame(json_data['questionChoiceList'])
    question_choices_df = question_choices_df['choiceScore']
    temp_df = pd.DataFrame(question_choices_df).transpose()
    temp_df['USER_TESTING_NO'] = user_testing_no
    self_report_df = pd.concat([self_report_df, temp_df], ignore_index=True)
print(self_report_df)
# 컬럼명 변경
self_report_df.columns = [f'q{col+1}' if isinstance(col, int) else col for col in self_report_df.columns]
# 컬럼명 순서 변경
self_report_df = self_report_df[['USER_TESTING_NO'] + [col for col in self_report_df.columns if col != 'USER_TESTING_NO']]
print(self_report_df)

    0  1  2  3  4  5  6  7  8  9  ... 108 109 110 111 112 113 114 115 116  \
0   1  4  3  3  3  3  3  3  1  1  ...   5   4   4   4   3   1   4   3   3   
1   4  2  3  3  1  3  3  5  4  3  ...   5   4   3   4   3   5   5   4   5   
2   3  3  3  3  1  3  3  3  2  4  ...   3   1   5   5   4   4   3   5   5   
3   4  5  5  2  1  1  1  3  4  3  ...   5   5   2   2   1   1   2   3   1   
4   4  5  3  4  3  3  1  2  1  1  ...   1   5   1   3   2   1   1   2   1   
5   4  5  5  5  2  5  2  4  5  3  ...   1   1   3   1   3   1   2   2   1   
6   3  4  4  2  3  3  4  4  5  5  ...   3   5   5   5   3   2   5   3   4   
7   5  5  2  5  3  4  4  4  4  3  ...   2   4   1   3   3   3   3   2   1   
8   1  5  5  4  4  2  2  1  3  3  ...   3   5   5   5   5   2   5   3   4   
9   2  1  1  2  1  1  2  3  3  2  ...   2   1   1   3   5   1   1   2   3   
10  3  3  5  5  5  3  4  3  1  5  ...   5   2   5   2   1   5   1   3   1   
11  3  3  3  1  2  3  2  2  1  4  ...   4   5   4   3   3   2   3   5   4   

In [82]:
result_self_scale_df = df.loc[:, ['target_code', 'user_testing_no']].copy().drop_duplicates(subset=['target_code', 'user_testing_no'])
for scale, items in self_scale_dict.items() :
    self_report_df[items] = self_report_df[items].apply(pd.to_numeric, errors='coerce')
    self_report_df[f'{scale}_평균점수'] = self_report_df[items].mean(axis=1).round(2)
    temp_df = self_report_df[['USER_TESTING_NO', f'{scale}_평균점수']].copy()
    result_self_scale_df = pd.merge(result_self_scale_df, temp_df, left_on='user_testing_no', right_on='USER_TESTING_NO', how='left')
    result_self_scale_df.drop(columns=['USER_TESTING_NO'], inplace=True)
result_self_scale_df

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37428\3251511194.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self_report_df[f'{scale}_평균점수'] = self_report_df[items].mean(axis=1).round(2)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37428\3251511194.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self_report_df[f'{scale}_평균점수'] = self_report_df[items].mean(axis=1).round(2)


,target_code,user_testing_no,사회적 발달 목표_평균점수,사회적 과시 목표_평균점수,인지적 공감_평균점수,정서적 공감_평균점수,도덕적 해이_평균점수,또래 동조성_평균점수,내재화_평균점수,행동적 수업 참여_평균점수,정서적 수업 참여_평균점수,발표 불안_평균점수,협력적 태도_평균점수,경쟁적 태도_평균점수
0,AD00012025115D1E4020,8916207158814a67abc2,4.00,3.83,4.29,4.14,2.71,3.00,1.7,2.4,3.6,2.14,4.00,4.12
1,AD00012025115D1E4019,a28c9553672c4f54ad7d,1.50,2.83,2.00,1.57,3.93,2.25,2.5,2.8,2.0,1.57,1.86,4.38
2,AD00012025115D1E4018,157327d769394633811f,2.67,2.67,3.00,3.43,2.79,3.92,2.4,2.0,3.4,1.71,2.57,2.75
3,AD00012025115D1E4017,d163f8c83ef64c50aa87,3.00,2.00,2.71,3.29,2.07,4.25,3.7,1.6,2.2,4.29,2.43,1.25
4,AD00012025115D1E4016,ac181f9b8eaa426cb4c8,4.00,4.17,4.43,3.43,1.93,2.92,1.9,4.0,3.8,1.43,4.29,3.88
5,AD00012025115D1E4015,72559204bbdf448b912f,2.50,1.83,3.14,2.71,2.43,3.83,3.9,3.0,2.4,3.71,2.57,1.50
6,AD00012025115D1E4014,30aa23408f104d1293a4,4.33,3.50,2.71,1.57,2.14,1.58,3.1,4.6,2.4,2.86,2.29,4.50
7,AD00012025115D1E4013,4f3f46fc2d2244758a78,3.17,4.33,3.00,1.43,1.86,4.08,3.1,2.0,3.8,3.43,2.14,3.88
8,AD00012025115D1E4012,dc9998566c234f0599c3,3.83,2.67,3.43,3.57,2.64,4.17,3.0,2.8,3.0,2.14,2.43,3.00
9,AD00012025115D1E4011,1eb4ec746358472a9a97,3.67,1.17,2.14,1.71,1.64,1.83,4.0,3.8,2.6,4.86,1.29,2.00


In [83]:
result_df = pd.merge(result_point_scale_df, result_self_scale_df, left_on=['target_code', 'user_testing_no'], right_on=['target_code', 'user_testing_no'], how='inner')
result_df

,target_code,user_testing_no,인기_지명수,선호_지명수,주도적 공격성_지명수,반응적 공격성_지명수,친사회성_지명수,가해_지명수,피해_지명수,사회적 발달 목표_평균점수,...,인지적 공감_평균점수,정서적 공감_평균점수,도덕적 해이_평균점수,또래 동조성_평균점수,내재화_평균점수,행동적 수업 참여_평균점수,정서적 수업 참여_평균점수,발표 불안_평균점수,협력적 태도_평균점수,경쟁적 태도_평균점수
0,AD00012025115D1E4020,8916207158814a67abc2,5,9,2,0,15,1,0,4.00,...,4.29,4.14,2.71,3.00,1.7,2.4,3.6,2.14,4.00,4.12
1,AD00012025115D1E4019,a28c9553672c4f54ad7d,0,0,5,3,1,15,0,1.50,...,2.00,1.57,3.93,2.25,2.5,2.8,2.0,1.57,1.86,4.38
2,AD00012025115D1E4018,157327d769394633811f,2,7,0,0,18,1,16,2.67,...,3.00,3.43,2.79,3.92,2.4,2.0,3.4,1.71,2.57,2.75
3,AD00012025115D1E4017,d163f8c83ef64c50aa87,0,0,1,10,2,0,12,3.00,...,2.71,3.29,2.07,4.25,3.7,1.6,2.2,4.29,2.43,1.25
4,AD00012025115D1E4016,ac181f9b8eaa426cb4c8,7,3,0,1,13,0,0,4.00,...,4.43,3.43,1.93,2.92,1.9,4.0,3.8,1.43,4.29,3.88
5,AD00012025115D1E4015,72559204bbdf448b912f,0,0,0,1,2,1,17,2.50,...,3.14,2.71,2.43,3.83,3.9,3.0,2.4,3.71,2.57,1.50
6,AD00012025115D1E4014,30aa23408f104d1293a4,0,0,0,0,0,1,0,4.33,...,2.71,1.57,2.14,1.58,3.1,4.6,2.4,2.86,2.29,4.50
7,AD00012025115D1E4013,4f3f46fc2d2244758a78,7,6,0,0,0,15,1,3.17,...,3.00,1.43,1.86,4.08,3.1,2.0,3.8,3.43,2.14,3.88
8,AD00012025115D1E4012,dc9998566c234f0599c3,7,4,2,0,13,1,20,3.83,...,3.43,3.57,2.64,4.17,3.0,2.8,3.0,2.14,2.43,3.00
9,AD00012025115D1E4011,1eb4ec746358472a9a97,0,0,0,0,0,0,2,3.67,...,2.14,1.71,1.64,1.83,4.0,3.8,2.6,4.86,1.29,2.00


In [84]:
result_temp_df = result_df[['target_code', 'user_testing_no']].copy()
for scale, items in point_scale_dict.items() :
    print(scale)
    print(items)
    print(result_df[f'{scale}_지명수'])
    temp_df = result_df[['target_code', 'user_testing_no']].copy()
    scale_mean = result_df[f'{scale}_지명수'].mean()
    temp_df[f'{scale}_z점수'] = (result_df[f'{scale}_지명수'] - scale_mean) / result_df[f'{scale}_지명수'].std()
    # 정규화
    def normalize(z):
        if z >= 0.84:
            return 5
        elif z >= 0.25:
            return 4
        elif z >= -0.25:
            return 3
        elif z >= -0.84:
            return 2
        else:
            return 1
    temp_df[f'{scale}_정규화'] = temp_df[f'{scale}_z점수'].apply(normalize)
    result_temp_df = pd.merge(result_temp_df, temp_df[['target_code', 'user_testing_no', f'{scale}_정규화']], on=['target_code', 'user_testing_no'], how='left')
result_temp_df

인기
[1]
0     5
1     0
2     2
3     0
4     7
5     0
6     0
7     7
8     7
9     0
10    6
11    7
12    0
13    1
14    4
15    0
16    0
17    1
18    0
19    2
Name: 인기_지명수, dtype: int64
선호
[2]
0     9
1     0
2     7
3     0
4     3
5     0
6     0
7     6
8     4
9     0
10    0
11    3
12    0
13    3
14    0
15    0
16    4
17    0
18    0
19    3
Name: 선호_지명수, dtype: int64
주도적 공격성
[4]
0     2
1     5
2     0
3     1
4     0
5     0
6     0
7     0
8     2
9     0
10    4
11    0
12    4
13    0
14    6
15    0
16    0
17    8
18    0
19    5
Name: 주도적 공격성_지명수, dtype: int64
반응적 공격성
[5]
0      0
1      3
2      0
3     10
4      1
5      1
6      0
7      0
8      0
9      0
10     5
11     0
12     4
13     0
14     6
15     0
16     0
17     8
18     0
19     1
Name: 반응적 공격성_지명수, dtype: int64
친사회성
[6, 7]
0     15
1      1
2     18
3      2
4     13
5      2
6      0
7      0
8     13
9      0
10     0
11     0
12     0
13     3
14     1
15     0
16    13
17     0
18     0
1

,target_code,user_testing_no,인기_정규화,선호_정규화,주도적 공격성_정규화,반응적 공격성_정규화,친사회성_정규화,가해_정규화,피해_정규화
0,AD00012025115D1E4020,8916207158814a67abc2,5,5,3,2,5,2,2
1,AD00012025115D1E4019,a28c9553672c4f54ad7d,2,2,5,4,2,5,2
2,AD00012025115D1E4018,157327d769394633811f,3,5,2,2,5,2,5
3,AD00012025115D1E4017,d163f8c83ef64c50aa87,2,2,2,5,2,2,5
4,AD00012025115D1E4016,ac181f9b8eaa426cb4c8,5,4,2,2,5,2,2
5,AD00012025115D1E4015,72559204bbdf448b912f,2,2,2,2,2,2,5
6,AD00012025115D1E4014,30aa23408f104d1293a4,2,2,2,2,2,2,2
7,AD00012025115D1E4013,4f3f46fc2d2244758a78,5,5,2,2,2,5,2
8,AD00012025115D1E4012,dc9998566c234f0599c3,5,4,3,2,5,2,5
9,AD00012025115D1E4011,1eb4ec746358472a9a97,2,2,2,2,2,2,2


In [85]:
result_df = pd.merge(result_temp_df, result_self_scale_df, left_on=['target_code', 'user_testing_no'], right_on=['target_code', 'user_testing_no'], how='inner')
result_df

,target_code,user_testing_no,인기_정규화,선호_정규화,주도적 공격성_정규화,반응적 공격성_정규화,친사회성_정규화,가해_정규화,피해_정규화,사회적 발달 목표_평균점수,...,인지적 공감_평균점수,정서적 공감_평균점수,도덕적 해이_평균점수,또래 동조성_평균점수,내재화_평균점수,행동적 수업 참여_평균점수,정서적 수업 참여_평균점수,발표 불안_평균점수,협력적 태도_평균점수,경쟁적 태도_평균점수
0,AD00012025115D1E4020,8916207158814a67abc2,5,5,3,2,5,2,2,4.00,...,4.29,4.14,2.71,3.00,1.7,2.4,3.6,2.14,4.00,4.12
1,AD00012025115D1E4019,a28c9553672c4f54ad7d,2,2,5,4,2,5,2,1.50,...,2.00,1.57,3.93,2.25,2.5,2.8,2.0,1.57,1.86,4.38
2,AD00012025115D1E4018,157327d769394633811f,3,5,2,2,5,2,5,2.67,...,3.00,3.43,2.79,3.92,2.4,2.0,3.4,1.71,2.57,2.75
3,AD00012025115D1E4017,d163f8c83ef64c50aa87,2,2,2,5,2,2,5,3.00,...,2.71,3.29,2.07,4.25,3.7,1.6,2.2,4.29,2.43,1.25
4,AD00012025115D1E4016,ac181f9b8eaa426cb4c8,5,4,2,2,5,2,2,4.00,...,4.43,3.43,1.93,2.92,1.9,4.0,3.8,1.43,4.29,3.88
5,AD00012025115D1E4015,72559204bbdf448b912f,2,2,2,2,2,2,5,2.50,...,3.14,2.71,2.43,3.83,3.9,3.0,2.4,3.71,2.57,1.50
6,AD00012025115D1E4014,30aa23408f104d1293a4,2,2,2,2,2,2,2,4.33,...,2.71,1.57,2.14,1.58,3.1,4.6,2.4,2.86,2.29,4.50
7,AD00012025115D1E4013,4f3f46fc2d2244758a78,5,5,2,2,2,5,2,3.17,...,3.00,1.43,1.86,4.08,3.1,2.0,3.8,3.43,2.14,3.88
8,AD00012025115D1E4012,dc9998566c234f0599c3,5,4,3,2,5,2,5,3.83,...,3.43,3.57,2.64,4.17,3.0,2.8,3.0,2.14,2.43,3.00
9,AD00012025115D1E4011,1eb4ec746358472a9a97,2,2,2,2,2,2,2,3.67,...,2.14,1.71,1.64,1.83,4.0,3.8,2.6,4.86,1.29,2.00


In [86]:
print(result_df)

             target_code       user_testing_no  인기_정규화  선호_정규화  주도적 공격성_정규화  \
0   AD00012025115D1E4020  8916207158814a67abc2       5       5            3   
1   AD00012025115D1E4019  a28c9553672c4f54ad7d       2       2            5   
2   AD00012025115D1E4018  157327d769394633811f       3       5            2   
3   AD00012025115D1E4017  d163f8c83ef64c50aa87       2       2            2   
4   AD00012025115D1E4016  ac181f9b8eaa426cb4c8       5       4            2   
5   AD00012025115D1E4015  72559204bbdf448b912f       2       2            2   
6   AD00012025115D1E4014  30aa23408f104d1293a4       2       2            2   
7   AD00012025115D1E4013  4f3f46fc2d2244758a78       5       5            2   
8   AD00012025115D1E4012  dc9998566c234f0599c3       5       4            3   
9   AD00012025115D1E4011  1eb4ec746358472a9a97       2       2            2   
10  AD00012025115D1E4010  a511507acce04b908351       5       2            5   
11  AD00012025115D1E4009  14c8df44e4004885a5e0      